*This notebook is not important*
___
# Testing

In [382]:
import numpy as np
from scipy.optimize import minimize
np.set_printoptions(threshold=300)
np.set_printoptions(suppress=True)

In [383]:
def sim_factor_model(loadings, specific_variance_vec, mu, nsim=1, verbose=True):
    """
    Parameters
    ---
    loadings:           (p, k) matrix
        Factorloadings

    specific_variance:  (p, p) matrix
        diagonal matrix of specific variances

    mu:                 (p, 1) vector 
        Means vector

    nsim:               float 
        How many observations should be simulated

    verbose:            boolean
        whether to print k and p

    Returns
    ---
        (n, p) matrix of observations from the specified factor model

    """
    k = loadings.shape[1]
    p = len(specific_variance_vec)

    if verbose:
        print(f"{k=} {p=}")

    # Generate nsim factor vectors from a standard normal distribution
    factor_vectors = np.random.normal(loc=0, scale=1, size=(nsim, k))

    # Generate nsim specific errors using element-wise normal sampling
    specific_errors = np.random.normal(loc=0, scale=np.sqrt(specific_variance_vec), size=(nsim, p))

    # Compute observations
    X = factor_vectors @ loadings.T + specific_errors + mu

    return X

In [ ]:
X = sim_factor_model(
    loadings=np.array([[10, 10, 10, 10, 0, 0, 0, 0],
                       [0, 0, 0, 0, 10, 10, 10, 10]]).T,
    specific_variance_vec=np.array([1 for _ in range(8)]),
    mu=np.array([1, 2, 3, 4, 5, 6, 7, 8]),
    nsim=6
)
print(X.shape, "\n", X)

k=2 p=8
(6, 8) 
 [[  8.45703888  10.23250852   9.57183238  11.70542423 -12.64747466
   -9.20166065  -6.3402936   -7.52827562]
 [  4.76745885   2.75364495   3.71766616   6.81339712   7.0111429
    7.83899223   6.71232947   7.59997821]
 [-11.7373603  -12.58988477  -9.28679286 -11.1618872    5.10749472
    8.35764699   8.75728496   8.2957431 ]
 [  6.6027086    7.06557176   7.55063417   9.73401983  16.57371113
   18.69412662  18.58836086  19.36711912]
 [  8.01441543   8.02764631   8.4255821   10.3181232   -6.10289401
   -1.89733244  -1.12323106  -2.059641  ]
 [-17.07015101 -16.81269825 -16.09470314 -16.38438441   2.89975474
    6.29755463   3.38934733   6.81701347]]


In [385]:
R = np.corrcoef(X, rowvar=False)
np.linalg.inv(R)

LinAlgError: Singular matrix

In [ ]:
np.linalg.inv(R + 0.1 * np.eye(8))

array([[ 7.52920976, -2.4993956 , -2.40283562, -2.38018884, -0.16878569,
         0.06687084,  0.03204021,  0.0753083 ],
       [-2.4993956 ,  7.39575085, -2.28239738, -2.31409344, -0.27584845,
         0.09118426,  0.02178631, -0.00741956],
       [-2.40283562, -2.28239738,  7.31259948, -2.4285773 , -0.03127653,
         0.12567476, -0.06126542,  0.23194592],
       [-2.38018884, -2.31409344, -2.4285773 ,  7.4584264 , -0.12002981,
        -0.25344282,  0.00143477,  0.01018863],
       [-0.16878569, -0.27584845, -0.03127653, -0.12002981,  7.52361588,
        -2.23384106, -2.45675874, -2.4368937 ],
       [ 0.06687084,  0.09118426,  0.12567476, -0.25344282, -2.23384106,
         7.22806114, -2.28097673, -2.46065061],
       [ 0.03204021,  0.02178631, -0.06126542,  0.00143477, -2.45675874,
        -2.28097673,  7.39868723, -2.4221024 ],
       [ 0.0753083 , -0.00741956,  0.23194592,  0.01018863, -2.4368937 ,
        -2.46065061, -2.4221024 ,  7.49311088]])

In [ ]:
def calculate_objective(specific_variance, X_data, k, standardized=True):
    """
    Calculate the factor model maximum likelihood objective function, F.

    Parameters
    ---
    specific_variance : (p,) arraylike
        The specific variances for each variable
    
    X_data : (n, p) arraylike

    k: float
        Number of factors

    standardized :       boolean
        Whether to use correlation matrix (standardized variables) or the covariance matrix
        in calculations.

    Returns
    ---
    Objective function value: float
    """
    p = X_data.shape[1]

    # Step 1
    S = np.corrcoef(X_data.T) if standardized else np.cov(X_data.T)
    Psi = np.diag(specific_variance)
    Psi_sq_inv = np.linalg.inv(Psi ** 0.5)
    S_star = Psi_sq_inv @ S @ Psi_sq_inv

    # Step 2
    eigval, eigvec = np.linalg.eig(S_star)

    # Step 3
    lambda_star = []
    for i in range(k):
        lambda_star.append(max(eigval[i] - 1, 0) ** 0.5 * eigvec[:,i])
    lambda_star = np.array(lambda_star).T

    # Step 4
    lambda_hat = Psi ** 0.5 @ lambda_star

    # Step 5
    internal_trace = np.linalg.inv(lambda_hat @ lambda_hat.T + Psi) @ S
    internal_log =np.linalg.inv(lambda_hat @ lambda_hat.T + Psi) @ (S + .1 * np.eye(p))
    result = np.trace(internal_trace) - np.log(np.linalg.det(internal_log)) - p

    return result

def factor_model_solution(X, k, x0_guess=None, standardized=True):
    """
    Optimize the factor model w.r.t. psi, and calculate psi hat and lambda hat.

    Parameters
    ---
    X: (n, p) arraylike
        data matrix

    k: integer
        Number of factors

    x0_guess: (p,) arraylike
        An inital guess for the minimization algorithm
        If x0_guess is None,
        then default guess is specific variance 1 for all variables.

    standardized:       boolean
        Whether to use correlation matrix (standardized variables) or the covariance matrix
        in calculations.

    Returns
    ---
    tuple : (psi_hat, lambda_hat) 
        where
        psi_hat: (p,p) diagonal matrix. 
        lambda_hat: (p, k) factor loadings matrix
    
    """
    
    if x0_guess == None:
        x0_guess = np.ones(X.shape[1])

    # Optimize
    problem = minimize(fun=lambda x: calculate_objective(np.exp(x), X_data=X, k=k, standardized=standardized),
                       x0=x0_guess)
    
    psi_hat = np.diag(np.exp(problem.x))

    # Calculate lambda hat
    S = np.corrcoef(X.T) if standardized else np.cov(X.T)
    Psi_sq_inv = np.linalg.inv(psi_hat ** 0.5)
    S_star = Psi_sq_inv @ S @ Psi_sq_inv
    eigval, eigvec = np.linalg.eig(S_star)
    lambda_star = []
    for i in range(k):
        lambda_star.append(max(eigval[i] - 1, 0) ** 0.5 * eigvec[:,i])
    lambda_star = np.array(lambda_star).T
    lambda_hat = psi_hat ** 0.5 @ lambda_star

    return (psi_hat, lambda_hat)

In [ ]:
factor_model_solution(X, 2, standardized=False)

c:\Users\Alexc\AppData\Local\Programs\Python\Python311\Lib\site-packages\scipy\optimize\_numdiff.py:598: ComplexWarning: Casting complex values to real discards the imaginary part
  J_transposed[i] = df / dx


(array([[0.        , 0.        , 0.        , 0.        , 0.        ,
         0.        , 0.        , 0.        ],
        [0.        , 0.12185591, 0.        , 0.        , 0.        ,
         0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.56449419, 0.        , 0.        ,
         0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        , 0.45337388, 0.        ,
         0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        , 0.        , 0.10030754,
         0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        , 0.        , 0.        ,
         0.39324569, 0.        , 0.        ],
        [0.        , 0.        , 0.        , 0.        , 0.        ,
         0.        , 0.27439525, 0.        ],
        [0.        , 0.        , 0.        , 0.        , 0.        ,
         0.        , 0.        , 0.00001761]]),
 array([[-12.74980917,   0.00000767],
        [-12.4380651 ,  -0.22460408],
  

In [ ]:
from utils import calculate_s
calculate_s(8, 2)

13.0